# DisasterAI — QLoRA İnce Ayar (Kaggle sürümü)

TÜBİTAK 2209-A / İP2. Taban model: `ytu-ce-cosmos/Turkish-Llama-8b-Instruct-v0.1`,
veri: 2.500 örneklik doğrulanmış corpus (train 2014 / val 251 / test 235).

**Kaggle kurulumu (sağ paneldeki Notebook options):**
1. **Accelerator → GPU T4 x2** (ya da P100) — GPU seçili olmadan çalışmaz.
2. **Internet → On** (model indirmek için şart).
3. **Input → Add Input → Upload → Dataset** ile `corpora_egitim.zip` içindeki
   üç jsonl dosyasını yükle (dataset adı: `disasterai-corpora`).

**Çalıştırma:** Hücreleri sırayla deneyebilirsin ama asıl yol sağ üstteki
**"Save Version → Save & Run All (Commit)"** — defter arka planda çalışır,
tarayıcıyı kapatabilirsin. Bitince Output sekmesinden adaptörü indirirsin.

In [ ]:
#@title 1) Kurulum (~3 dk)
!pip -q install -U "transformers>=4.44" "trl>=0.9" "peft>=0.12" "bitsandbytes>=0.43" "datasets>=2.20" accelerate
print("kurulum tamam")

In [ ]:
#@title 2) Veriyi bul ve yükle
import glob, os, json
from datasets import load_dataset

aday = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)
if not aday:
    raise SystemExit("train.jsonl bulunamadi. Sag panel > Add Input > Upload ile "
                     "train/val/test.jsonl dosyalarini dataset olarak ekle.")
VERI_DIR = os.path.dirname(aday[0])
print("veri dizini:", VERI_DIR)

ds = load_dataset("json", data_files={
    "train": f"{VERI_DIR}/train.jsonl",
    "val":   f"{VERI_DIR}/val.jsonl",
})
print(ds)
print("ornek anahtarlar:", ds["train"][0].keys())

In [ ]:
#@title 3) Sohbet biçimine çevir
def bicimlendir(kayit):
    return {"messages": [
        {"role": "user", "content": kayit["prompt"]},
        {"role": "assistant", "content": kayit["cikti"]},
    ]}

train_ds = ds["train"].map(bicimlendir, remove_columns=ds["train"].column_names)
val_ds   = ds["val"].map(bicimlendir,   remove_columns=ds["val"].column_names)
print("train:", len(train_ds), "| val:", len(val_ds))

In [ ]:
#@title 4) Taban modeli 4-bit yukle — TEK GPU (~5-10 dk)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # Kaggle T4 x2 -> tek karta sabitle

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "ytu-ce-cosmos/Turkish-Llama-8b-Instruct-v0.1"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map={"": 0}   # bolme yok
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("model hazir (tek GPU):", MODEL_ID, "| cihaz:", next(model.parameters()).device)

In [ ]:
#@title 5) LoRA + egitim ayarlari (checkpoint'li, surum-uyumlu)
import inspect, torch, trl
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
print("trl surumu:", trl.__version__)

CIKTI_DIR = "/kaggle/working/qlora_cikti"

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules="all-linear", task_type="CAUSAL_LM",
)

istenen = dict(
    output_dir=CIKTI_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,
    fp16=True,
    max_length=1024,
    packing=False,
    report_to="none",
)
gecerli = set(inspect.signature(SFTConfig.__init__).parameters)
kullanilan = istenen if "kwargs" in gecerli else {k: v for k, v in istenen.items() if k in gecerli}
atlanan = sorted(set(istenen) - set(kullanilan))
if atlanan:
    print("bu surumde olmayan ayarlar atlandi:", atlanan)
ayar = SFTConfig(**kullanilan)

tr_gecerli = set(inspect.signature(SFTTrainer.__init__).parameters)
ek = {"processing_class": tokenizer} if "processing_class" in tr_gecerli else {"tokenizer": tokenizer}
egitici = SFTTrainer(model=model, args=ayar,
                     train_dataset=train_ds, eval_dataset=val_ds,
                     peft_config=lora, **ek)

# T4 uyumu: egitilebilir (LoRA) parametreleri fp32'ye sabitle
n = 0
for p in egitici.model.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.float(); n += 1
print(f"{n} parametre fp32'ye sabitlendi — egitici hazir")

In [ ]:
#@title 6) EGITIM (~2-3 saat; Save & Run All ile arka planda calisir)
import glob
ckler = sorted(glob.glob(f"{CIKTI_DIR}/checkpoint-*"),
               key=lambda p: int(p.split("-")[-1]))
if ckler:
    print("checkpoint bulundu, devam:", ckler[-1])
    egitici.train(resume_from_checkpoint=ckler[-1])
else:
    print("sifirdan basliyor")
    egitici.train()
print("egitim bitti ✅")

In [ ]:
#@title 7) Adaptoru kaydet (Kaggle Output'a)
KAYIT = "/kaggle/working/disasterai_lora"
egitici.model.save_pretrained(KAYIT)
tokenizer.save_pretrained(KAYIT)
import os
print("kaydedildi:", KAYIT)
print(os.listdir(KAYIT))
print("\nDefter bitince sag paneldeki Output sekmesinden bu klasoru indir.")

In [ ]:
#@title 8) Hizli deneme — modelin ilk ciktilari
import json, glob, torch

test_yolu = glob.glob("/kaggle/input/**/test.jsonl", recursive=True)[0]
ornekler = [json.loads(s) for s in open(test_yolu)][:3]

egitici.model.eval()
for o in ornekler:
    mesajlar = [{"role": "user", "content": o["prompt"]}]
    girdi = tokenizer.apply_chat_template(mesajlar, add_generation_prompt=True,
                                          return_tensors="pt").to(egitici.model.device)
    with torch.no_grad():
        cikti = egitici.model.generate(**girdi, max_new_tokens=400,
                                       temperature=0.7, do_sample=True,
                                       pad_token_id=tokenizer.eos_token_id)
    yanit = tokenizer.decode(cikti[0][girdi.shape[1]:], skip_special_tokens=True)
    print("=" * 70)
    print(f"[{o['ton']} | {o['afet_tipi']}]")
    print(yanit.strip()[:600])